In [1]:
import pandas as pd
import numpy as np

### Load data

In [14]:
td = pd.read_csv('../../data/Kim_thickdisk.csv')
df_tic = pd.read_csv('../../data/TIC_catalog_data_all_120s.csv')
td_xm = pd.read_csv('../../data/Kim_thickdisk_dr3_dr2_xmatch.csv')
tic_td = pd.read_csv('../../data/TIC_thickdisk.csv')

/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_43074/2907735732.py:2: DtypeWarning: Columns (0: TYC, 1: UCAC, 2: TWOMASS, 3: ALLWISE, 4: TWOMflag, 5: BmagFlag) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tic = pd.read_csv('../../data/TIC_catalog_data_all_120s.csv')


### Cross-match EDR3 → DR2 via `gaiadr3.dr2_neighbourhood`

Match `td.GaiaEDR3` against `dr3_source_id` in the Gaia crossmatch table.
EDR3 and DR3 source IDs are identical, so this table is the right one even
though we're starting from EDR3 IDs. Batched IN-list queries because the
full list (~550k IDs) exceeds any single ADQL call's URL length. All rows
in `td` are kept via left join so duplicate DR2 matches surface as
multiple output rows per EDR3 id.

In [7]:
from astroquery.gaia import Gaia

CHUNK = 5000
edr3_ids = td['GaiaEDR3'].astype('int64').unique()
print(f'{len(edr3_ids):,} unique EDR3 source ids to cross-match')

frames = []
n_batches = (len(edr3_ids) + CHUNK - 1) // CHUNK
for b, start in enumerate(range(0, len(edr3_ids), CHUNK), start=1):
    batch = edr3_ids[start:start + CHUNK]
    id_list = ','.join(map(str, batch))
    adql = f"""
    SELECT dr3_source_id, dr2_source_id, angular_distance,
           magnitude_difference, proper_motion_propagation
    FROM gaiadr3.dr2_neighbourhood
    WHERE dr3_source_id IN ({id_list})
    """
    job = Gaia.launch_job_async(adql, dump_to_file=False)
    res = job.get_results().to_pandas()
    frames.append(res)
    print(f'  batch {b}/{n_batches}: {len(res):,} rows')

xmatch = pd.concat(frames, ignore_index=True)
print(f'\nTotal crossmatch rows: {len(xmatch):,}')
print(f'Distinct dr3_source_ids returned: {xmatch["dr3_source_id"].nunique():,}')

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).
551,214 unique EDR3 source ids to cross-match
INFO: Query finished. [astroquery.utils.tap.core]
  batch 1/111: 5,244 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 2/111: 5,147 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 3/111: 5,090 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 4/111: 5,077 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 5/111: 5,114 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 6/111: 5,072 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 7/111: 5,076 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 

RemoteDisconnected: Remote end closed connection without response

In [8]:
import time

# Resume the batched query from wherever the previous cell crashed.
# frames is preserved in the kernel; everything successfully appended is intact.
assert 'frames' in dir() and 'edr3_ids' in dir(), \
    'frames or edr3_ids missing — rerun the initial query cell first'
n_done = len(frames)
rows_done = sum(len(f) for f in frames)
print(f'completed batches: {n_done}/{n_batches}  ({rows_done:,} rows in memory)')

remaining_start = n_done * CHUNK
if remaining_start >= len(edr3_ids):
    print('all batches already complete — skipping resume loop')
else:
    print(f'resuming from edr3_ids index {remaining_start:,} of {len(edr3_ids):,}\n')
    for b, start in enumerate(range(remaining_start, len(edr3_ids), CHUNK),
                              start=n_done + 1):
        batch = edr3_ids[start:start + CHUNK]
        id_list = ','.join(map(str, batch))
        adql = f"""
        SELECT dr3_source_id, dr2_source_id, angular_distance,
               magnitude_difference, proper_motion_propagation
        FROM gaiadr3.dr2_neighbourhood
        WHERE dr3_source_id IN ({id_list})
        """
        for attempt in range(3):
            try:
                job = Gaia.launch_job_async(adql, dump_to_file=False)
                res = job.get_results().to_pandas()
                break
            except Exception as e:
                wait = 5 * 2 ** attempt
                print(f'  batch {b} attempt {attempt+1} failed: {e!r}; sleeping {wait}s')
                time.sleep(wait)
        else:
            raise RuntimeError(f'batch {b} failed after 3 attempts')
        frames.append(res)
        print(f'  batch {b}/{n_batches}: {len(res):,} rows')

xmatch = pd.concat(frames, ignore_index=True)
print(f'\nTotal crossmatch rows: {len(xmatch):,}')
print(f'Distinct dr3_source_ids returned: {xmatch["dr3_source_id"].nunique():,}')

completed batches: 82/111  (419,112 rows in memory)
resuming from edr3_ids index 410,000 of 551,214

INFO: Query finished. [astroquery.utils.tap.core]
  batch 83/111: 5,042 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 84/111: 5,038 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 85/111: 5,027 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 86/111: 5,025 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 87/111: 5,078 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 88/111: 5,108 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 89/111: 5,056 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 90/111: 5,085 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 91/111: 5,048 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 92/111: 5,051 rows
INFO: Query finished. [astroquery.utils.tap.core]
  batch 93/111: 5,021 rows
INFO: Query finished. [astroquery.utils.tap.core]
  

In [9]:
td_xm = td.merge(
    xmatch,
    left_on='GaiaEDR3',
    right_on='dr3_source_id',
    how='left',
)

match_counts = td_xm.groupby('GaiaEDR3').size()
print(f'td rows           : {len(td):,}')
print(f'joined rows       : {len(td_xm):,}')
print(f'EDR3 ids unmatched: {(match_counts == 0).sum():,}  '
      f'(detected as NaN dr2_source_id)')
print(f'EDR3 ids with 1 match : {(match_counts == 1).sum():,}')
print(f'EDR3 ids with >1 match: {(match_counts > 1).sum():,}')

dups = td_xm[td_xm.duplicated('GaiaEDR3', keep=False)]
print(f'\nduplicate-match rows for inspection: {len(dups):,}')
dups.head(20)

td rows           : 551,214
joined rows       : 561,246
EDR3 ids unmatched: 0  (detected as NaN dr2_source_id)
EDR3 ids with 1 match : 541,550
EDR3 ids with >1 match: 9,664

duplicate-match rows for inspection: 19,696


,GaiaEDR3,RAdeg,DEdeg,plx,e_plx,pmRA,pmDE,Gmag,G-RP,E(G-RP),AG,[Fe/H]KNN,[Fe/H]grid,dr3_source_id,dr2_source_id,angular_distance,magnitude_difference,proper_motion_propagation
7,6255613166476392320,227.780304,-21.167114,2.7782,0.1480,-59.113,-29.145,18.180,1.130,0.073,0.299,-0.231,-0.576,6.255613e+18,6.255613e+18,1528.845703,-1.726631,True
8,6255613166476392320,227.780304,-21.167114,2.7782,0.1480,-59.113,-29.145,18.180,1.130,0.073,0.299,-0.231,-0.576,6.255613e+18,6.255613e+18,0.091682,-0.036106,True
40,5969553299294803072,252.992080,-41.227735,3.6755,0.2025,-113.501,-265.265,18.232,1.266,0.000,0.000,-0.184,0.300,5.969553e+18,5.969553e+18,1.355578,-0.007931,True
41,5969553299294803072,252.992080,-41.227735,3.6755,0.2025,-113.501,-265.265,18.232,1.266,0.000,0.000,-0.184,0.300,5.969553e+18,5.969553e+18,1405.043457,-2.159130,True
42,5969553299294803072,252.992080,-41.227735,3.6755,0.2025,-113.501,-265.265,18.232,1.266,0.000,0.000,-0.184,0.300,5.969553e+18,5.969553e+18,1607.654907,0.042061,True
44,4492070943011360128,261.185863,10.518933,3.5093,0.1277,-50.753,-144.715,17.917,1.251,0.110,0.449,-0.149,0.300,4.492071e+18,4.492071e+18,0.213513,-0.021156,True
45,4492070943011360128,261.185863,10.518933,3.5093,0.1277,-50.753,-144.715,17.917,1.251,0.110,0.449,-0.149,0.300,4.492071e+18,4.492071e+18,1320.705444,-1.449373,True
52,4491673710078571136,260.452868,9.992002,2.7261,0.0590,-5.504,-128.443,16.785,1.110,0.049,0.200,-0.151,0.300,4.491674e+18,4.491674e+18,0.118356,-0.020702,True
53,4491673710078571136,260.452868,9.992002,2.7261,0.0590,-5.504,-128.443,16.785,1.110,0.049,0.200,-0.151,0.300,4.491674e+18,4.491674e+18,1378.374268,-1.934448,True
71,4149071728064112000,265.631867,-14.720002,2.6145,0.1221,-24.424,-110.519,17.859,1.164,0.122,0.499,-0.144,0.083,4.149072e+18,4.149072e+18,1220.059204,-1.785513,True


In [17]:
# Pick the best DR2 counterpart per EDR3 id.
#   1. Apply quality cuts: angular_distance < SEP_MAX and |dmag| < DMAG_MAX.
#   2. Among rows that pass, the smallest angular distance wins
#      (ties broken by smallest |magnitude_difference|).
#   3. Label every joined row:
#        'best'                — the chosen DR2 match
#        'rejected_alternate'  — passes cuts but loses to a better match
#        'rejected_quality'    — has a DR2 entry but fails cuts
#        'unmatched'           — no DR2 row at all (left-join NaN)
SEP_MAX = 10.0   # arcsec  (angular_distance is in arcsec in gaiadr3.dr2_neighbourhood)
DMAG_MAX = 1.0   # mag

sep      = td_xm['angular_distance'].astype(float)
dmag_abs = td_xm['magnitude_difference'].abs()
has_match  = td_xm['dr2_source_id'].notna()
quality_ok = has_match & (sep < SEP_MAX) & (dmag_abs < DMAG_MAX)

td_xm['match_status'] = 'unmatched'
td_xm.loc[has_match, 'match_status'] = 'rejected_quality'

qa = td_xm[quality_ok].assign(_absdmag=dmag_abs.loc[quality_ok])
qa_sorted = qa.sort_values(['GaiaEDR3', 'angular_distance', '_absdmag'])
best_idx = qa_sorted.groupby('GaiaEDR3', sort=False).head(1).index

td_xm.loc[qa.index, 'match_status']  = 'rejected_alternate'
td_xm.loc[best_idx, 'match_status']  = 'best'

print('match_status counts (rows):')
print(td_xm['match_status'].value_counts(dropna=False).to_string())
print()
n_unique_best = td_xm.loc[td_xm['match_status'] == 'best', 'GaiaEDR3'].nunique()
n_no_best     = td_xm['GaiaEDR3'].nunique() - n_unique_best
print(f'distinct EDR3 ids with a chosen best : {n_unique_best:,}')
print(f'distinct EDR3 ids without any best   : {n_no_best:,}')

match_status counts (rows):
match_status
best                551165
rejected_quality     10075
unmatched                6

distinct EDR3 ids with a chosen best : 551,165
distinct EDR3 ids without any best   : 49


In [18]:
# Count EDR3 ids that still have >1 candidate after the quality cuts.
# Each such id has exactly one 'best' row and one or more 'rejected_alternate' rows.
# (Requires the selection cell above to have been re-run on the rebuilt td_xm so
#  match_status is present.)
assert 'match_status' in td_xm.columns, \
    'match_status missing — re-run the selection cell above on the rebuilt td_xm'

alt_per_id      = td_xm[td_xm['match_status'] == 'rejected_alternate'].groupby('GaiaEDR3').size()
n_ids_with_dups = int(len(alt_per_id))
n_alt_rows      = int(alt_per_id.sum())
print(f'EDR3 ids with >1 match passing cuts (sep<{SEP_MAX}, |dmag|<{DMAG_MAX}): {n_ids_with_dups:,}')
print(f'    total alternate (non-best) rows still in td_xm: {n_alt_rows:,}')

# Spot-check: show the top 10 such EDR3 ids with their best / alternate rows side by side
example_ids = alt_per_id.index[:10]
cols = ['GaiaEDR3', 'dr2_source_id', 'angular_distance',
        'magnitude_difference', 'proper_motion_propagation', 'match_status']
td_xm[td_xm['GaiaEDR3'].isin(example_ids)][cols].sort_values(
    ['GaiaEDR3', 'angular_distance']
)

EDR3 ids with >1 match passing cuts (sep<10.0, |dmag|<1.0): 0
    total alternate (non-best) rows still in td_xm: 0


,GaiaEDR3,dr2_source_id,angular_distance,magnitude_difference,proper_motion_propagation,match_status


In [26]:
td_xm.columns

Index(['GaiaEDR3', 'RAdeg', 'DEdeg', 'plx', 'e_plx', 'pmRA', 'pmDE', 'Gmag',
       'G-RP', 'E(G-RP)', 'AG', '[Fe/H]KNN', '[Fe/H]grid', 'dr3_source_id',
       'dr2_source_id', 'angular_distance', 'magnitude_difference',
       'proper_motion_propagation', 'match_status'],
      dtype='str')

In [7]:
td_xm.loc[td_xm['match_status'] == 'best'][['dr3_source_id','dr2_source_id']].to_csv('../../data/Kim_thickdisk_dr3_dr2_xmatch.csv', index=False)

KeyError: 'match_status'

### Filter to stars with 2min LCs in TIC catalogue

In [10]:
len(td_xm)

551165

In [18]:
gids_thick = td_xm['dr2_source_id'].astype(float).values
df_tic['thick'] = df_tic['GAIA'].isin(gids_thick)

/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_43074/895443286.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tic['thick'] = df_tic['GAIA'].isin(gids_thick)


In [19]:
tic_thick = df_tic.loc[(df_tic['thick'] == True)]

In [14]:
tic_thick.to_csv('../../data/TIC_thickdisk.csv', index=False)

In [20]:
joined = tic_thick.merge(td_xm, left_on='GAIA', right_on='dr2_source_id', how='left')

In [21]:
joined[['ID','dr3_source_id', 'dr2_source_id','Tmag']].to_csv('../../data/TIC_thickdisk_joined.csv', index=False)

In [5]:
tic_td[['ID','Tmag']].to_csv('../../data/TIC_thickdisk_Tmags.csv', index=False)

In [7]:
td_xm = td_xm.merge(tic_td[['ID','Tmag','GAIA']], left_on='dr2_source_id', right_on='GAIA', how='left', indicator=True)

In [10]:
tic_td[['ID','GAIA']]

,ID,GAIA
0,5422,6.223042e+18
1,38752,6.222569e+18
2,738200,4.880352e+18
3,75437,6.228871e+18
4,138659,6.229366e+18
...,...,...
6050,1984066557,4.230381e+18
6051,1989175422,6.466133e+18
6052,2028097683,6.611621e+18
6053,2050277302,2.213351e+18
